# Adaboost implementation
```
initialize weights
       ↓
train weak classifier
       ↓
measure its error
       ↓
compute classifier importance
       ↓
increase weight of mistakes
       ↓
train next classifier
       ↓
repeat
       ↓
weighted vote of all classifiers
```

In [1]:
import numpy as np

In [3]:
cd = float("inf")
cd

inf

In [5]:
class WeakLearner:
    def __init__(self):
        self.feature_index = None
        self.threshold     = None
        self.polarity      = None
     
    def fit(self, X, y, sample_weights):
        n_samples, n_features = X.shape
        best_error = float("inf")
        
        for feature in range(n_features):
            feature_values  = X[:,0]
            thresholds  = np.unique(feature_values)
            
            for t in thresholds:
                for  polarity in [-1,1]:
                    predictions = np.ones(n_samples)

                    if polarity == 1:
                        predictions[feature_values < t] = -1
                    else:
                        predictions[feature_values >= t] = -1
                        
                                        # Weighted classification error
                    misclassified = predictions != y

                    error = np.sum(
                        sample_weights[misclassified]
                    )

                    # Save best split
                    if error < best_error:
                        best_error = error
                        self.feature_index = feature
                        self.threshold = t
                        self.polarity = polarity
                        
                                        # Save best split
                    if error < best_error:
                        best_error = error
                        self.feature_index = feature
                        self.threshold = t
                        self.polarity = polarity            
        
        return best_error
    
    def predict(self, X):
        feature_values = X[:, self.feature_index]

        predictions = np.ones(X.shape[0])

        if self.polarity == 1:
            predictions[feature_values < self.threshold] = -1
        else:
            predictions[feature_values >= self.threshold] = -1

        return predictions

In [ ]:
data = np.array([[0.3,12,-3.0],
                 [0.15,10,-1.0],
                 [0.4,15,-2.5],
                 [0.2,9,-1.3],
                 [0.6,8,-2.3],
                 [0.2,5,-2.1],
                 [-0.3,9,-3.4],
                 [-0.4,3,-1.1]])

data_test  = np.array([[0.4,14,-2.7]])

y  = np.array([1,-1,-1,1,1,-1,-1,1])

weights = np.full(data.shape[0],1/(data.shape[0]))

stump =  WeakLearner()

error = stump.fit(data,y, weights)

preds = stump.predict(data_test)

print(data[:,0])
print("Feature:",     stump.feature_index)
print("Threshold:",   stump.threshold)
print("Error:",       error)
print("Predictions:", preds)

[ 0.3   0.15  0.4   0.2   0.6   0.2  -0.3  -0.4 ]
Feature: 0
Threshold: -0.3
Error: 0.375
Predictions: [-1.]


In [7]:
weighted_error = np.sum(weights * (preds != y))
weighted_error

np.float64(0.5)

In [11]:
eps = 1e-10  # avoids division by zero

weighted_error = np.clip(weighted_error, eps, 1 - eps)

alpha = 0.5 * np.log((1-weighted_error)/weighted_error)

In [ ]:
weights = weights * np.exp(-alpha*y*preds)
weights = weights / np.sum(weights)

weights

array([0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125])